In [18]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

In [19]:
# ============================================================
# CARGA DE DATOS
# ============================================================

CSV_FILE = "resultados_onpe_erm2022_por_distrito.csv"  # mismo CSV que generaste

df = pd.read_csv(CSV_FILE)

# Asegurarnos de que la columna de votos es numérica
df["total_votos"] = pd.to_numeric(df["total_votos"], errors="coerce").fillna(0)

# Limpiar textos por si acaso
for col in ["region", "provincia", "distrito", "organizacion_politica"]:
    df[col] = df[col].astype(str).str.strip()

# Opciones únicas para los dropdowns
regiones = sorted(df["region"].dropna().unique())


In [20]:
# ============================================================
# APLICACIÓN DASH
# ============================================================

app = Dash(__name__)

app.layout = html.Div([
    html.H2("Resultados ERM 2022 - Municipal distrital (ONPE)"),

    html.Div([
        html.Div([
            html.Label("Región"),
            dcc.Dropdown(
                id="region-dropdown",
                options=[{"label": r, "value": r} for r in regiones],
                placeholder="Todas las regiones",
                multi=False,
                clearable=True,
            ),
        ], style={"flex": "1", "margin-right": "10px"}),

        html.Div([
            html.Label("Provincia"),
            dcc.Dropdown(
                id="provincia-dropdown",
                placeholder="Todas las provincias",
                multi=False,
                clearable=True,
            ),
        ], style={"flex": "1", "margin-right": "10px"}),

        html.Div([
            html.Label("Distrito"),
            dcc.Dropdown(
                id="distrito-dropdown",
                placeholder="Todos los distritos",
                multi=False,
                clearable=True,
            ),
        ], style={"flex": "1"}),
    ], style={"display": "flex", "margin-bottom": "20px"}),

    dcc.Graph(id="grafico-votos"),
])

# ============================================================
# CALLBACK: ACTUALIZAR PROVINCIAS SEGÚN REGIÓN
# ============================================================

@app.callback(
    Output("provincia-dropdown", "options"),
    Output("provincia-dropdown", "value"),
    Input("region-dropdown", "value"),
)
def actualizar_provincias(region_sel):
    df_fil = df.copy()

    if region_sel:
        df_fil = df_fil[df_fil["region"] == region_sel]

    provincias = sorted(df_fil["provincia"].dropna().unique())
    options = [{"label": p, "value": p} for p in provincias]

    # resetea la selección de provincia cuando cambia la región
    return options, None

# ============================================================
# CALLBACK: ACTUALIZAR DISTRITOS SEGÚN REGIÓN Y PROVINCIA
# ============================================================

@app.callback(
    Output("distrito-dropdown", "options"),
    Output("distrito-dropdown", "value"),
    Input("region-dropdown", "value"),
    Input("provincia-dropdown", "value"),
)
def actualizar_distritos(region_sel, prov_sel):
    df_fil = df.copy()

    if region_sel:
        df_fil = df_fil[df_fil["region"] == region_sel]
    if prov_sel:
        df_fil = df_fil[df_fil["provincia"] == prov_sel]

    distritos = sorted(df_fil["distrito"].dropna().unique())
    options = [{"label": d, "value": d} for d in distritos]

    # resetea la selección de distrito cuando cambian región/provincia
    return options, None

# ============================================================
# CALLBACK PRINCIPAL: ACTUALIZAR GRÁFICA
# ============================================================

def wrap_text(text, width=14):
    """
    Corta un texto muy largo en varias líneas cada `width` caracteres.
    """
    import textwrap
    return "<br>".join(textwrap.wrap(text, width))

@app.callback(
    Output("grafico-votos", "figure"),
    Input("region-dropdown", "value"),
    Input("provincia-dropdown", "value"),
    Input("distrito-dropdown", "value"),
)
def actualizar_grafico(region_sel, prov_sel, dist_sel):
    df_fil = df.copy()

    # Filtros tipo pivot
    if region_sel:
        df_fil = df_fil[df_fil["region"] == region_sel]
    if prov_sel:
        df_fil = df_fil[df_fil["provincia"] == prov_sel]
    if dist_sel:
        df_fil = df_fil[df_fil["distrito"] == dist_sel]

    if df_fil.empty:
        fig = px.bar(title="Sin datos para el filtro seleccionado",
                     template="plotly_white")
        return fig

    # Agrupar resultados
    resumen = (
        df_fil.groupby("organizacion_politica")["total_votos"]
              .sum()
              .sort_values(ascending=False)
              .reset_index()
    )

    # *** MULTILÍNEA en etiquetas del eje X ***
    resumen["label_wrap"] = resumen["organizacion_politica"].apply(wrap_text)

    # TÍTULO
    titulo = "Votos por organización política"
    if region_sel:
        titulo += f" - Región: {region_sel}"
    if prov_sel:
        titulo += f" - Provincia: {prov_sel}"
    if dist_sel:
        titulo += f" - Distrito: {dist_sel}"

    # FIGURA
    fig = px.bar(
        resumen,
        x="label_wrap",
        y="total_votos",
        text="total_votos",
        color_discrete_sequence=["#4C72B0"],
        title=titulo,
        template="plotly_white",
    )

    # Etiquetas de texto
    fig.update_traces(
        textposition="outside",
        texttemplate="%{text:,}"
    )

    # LAYOUT
    fig.update_layout(
        showlegend=False,
        height=650,
        margin=dict(l=80, r=40, t=80, b=150),
        title=dict(
            x=0.5,
            xanchor="center",
            font=dict(size=22, family="Arial")
        ),
        xaxis=dict(
            title="Organización política",
            tickangle=0,                     # ← HORIZONTAL
            tickfont=dict(size=10),
            automargin=True,
        ),
        yaxis=dict(
            title="Total de votos",
            tickfont=dict(size=12),
            title_font=dict(size=16),
            showgrid=True,
            gridcolor="rgba(0,0,0,0.1)",
        )
    )

    return fig


In [21]:
# ============================================================
# EJECUCIÓN
# ============================================================

if __name__ == "__main__":
    app.run(debug=True)

In [26]:
# 1) Partido ganador por distrito
idx_max = df.groupby("ubigeo")["total_votos"].idxmax()
df_ganadores = df.loc[idx_max]

# 2) Conteo de distritos ganados por partido
conteo = (
    df_ganadores.groupby("organizacion_politica")["ubigeo"]
    .nunique()
    .reset_index(name="num_distritos_1er_lugar")
)

# ORDEN INVERSO (mayor → menor)
conteo = conteo.sort_values("num_distritos_1er_lugar", ascending=True)

# 3) Gráfico de barras horizontales
fig = px.bar(
    conteo,
    x="num_distritos_1er_lugar",
    y="organizacion_politica",
    orientation="h",
    text="num_distritos_1er_lugar",
    title="Distritos en donde cada partido ganó",
    template="plotly_white",
    color_discrete_sequence=["#4C72B0"],
)

# Texto fuera de las barras
fig.update_traces(
    textposition="outside",
    texttemplate="%{text}",
)

# Ajustes visuales finos
fig.update_layout(
    showlegend=False,
    height=1800,                          # necesario por la cantidad de partidos
    margin=dict(l=350, r=40, t=80, b=40),
    title=dict(
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    xaxis=dict(
        title="Número de distritos en 1er lugar",
        title_font=dict(size=16),
        tickfont=dict(size=12),
        showgrid=True,
        gridcolor="rgba(0,0,0,0.1)",
    ),
    yaxis=dict(
        title="Organización política",
        title_font=dict(size=16),
        tickfont=dict(size=9),            # ← LETRAS MÁS PEQUEÑAS
        automargin=True,
    ),
)

fig.show()

In [28]:
# Lista de partidos a conservar (debe coincidir exactamente con el Excel)
PARTIDOS_FILTRO = [
    "PARTIDO POLITICO FRENTE DE LA ESPERANZA 2021",
    "PARTIDO PATRIOTICO DEL PERU",
    "JUNTOS POR EL PERU",
    "FUERZA POPULAR",
    "ACCION POPULAR",
    "PODEMOS PERU",
    "RENOVACION POPULAR",
    "ALIANZA PARA EL PROGRESO",
    "AVANZA PAIS – PARTIDO DE INTEGRACION SOCIAL",
    "PARTIDO MORADO",
    "PARTIDO POLITICO NACIONAL PERU LIBRE",
    "PARTIDO DEMOCRÁTICO SOMOS PERU",
]

# Filtro aplicado al dataset
df_filtrado = df[df["organizacion_politica"].isin(PARTIDOS_FILTRO)].copy()

print("Filas antes:", len(df))
print("Filas después del filtro:", len(df_filtrado))

df_filtrado.head()

Filas antes: 8797
Filas después del filtro: 3058


,ubigeo,region,provincia,distrito,organizacion_politica,total_votos
1,10102,Amazonas,Chachapoyas,Asuncion,ALIANZA PARA EL PROGRESO,159
4,10103,Amazonas,Chachapoyas,Balsas,RENOVACION POPULAR,130
5,10103,Amazonas,Chachapoyas,Balsas,ALIANZA PARA EL PROGRESO,248
8,10104,Amazonas,Chachapoyas,Cheto,PARTIDO POLITICO NACIONAL PERU LIBRE,17
11,10105,Amazonas,Chachapoyas,Chiliquin,JUNTOS POR EL PERU,33


In [31]:
# ============================
# 1. Cargar datos
# ============================

df = pd.read_csv("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Scrapping/resultados_onpe_erm2022_por_distrito.csv")
tabla = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Scrapping/tabla_partidos.xlsx")

# Normalizar nombres (mayúsculas)
df["organizacion_politica"] = df["organizacion_politica"].str.upper()
tabla["organizacion_politica"] = tabla["organizacion_politica"].str.upper()
# ============================
# 2. Identificar ganadores por distrito
# ============================

# Elegir ganador por distrito según mayor cantidad de votos
ganadores = (
    df.sort_values(["ubigeo", "total_votos"], ascending=[True, False])
      .groupby("ubigeo")
      .first()
      .reset_index()
)
# ============================
# 3. Filtrar solo partidos del listado
# ============================

filtered = ganadores.merge(tabla, on="organizacion_politica", how="inner")

# ============================
# 4. Contar distritos ganados
# ============================

conteo = (
    filtered.groupby(["organizacion_politica", "ubicación"])
            .size()
            .reset_index(name="distritos_ganados")
            .sort_values("ubicación")
)
# ============================
# 5. Gráfico
# ============================

fig = px.scatter(
    conteo,
    x="ubicación",
    y="distritos_ganados",
    text="organizacion_politica",
    size="distritos_ganados",
    color="distritos_ganados",
    color_continuous_scale="Blues",
    title="Relación entre posición en la cédula y número de distritos ganados",
)

fig.update_traces(textposition="top center")

fig.update_layout(
    xaxis_title="Posición en la cédula",
    yaxis_title="Distritos ganados",
    font=dict(size=12),
    height=700,
)

fig.show()